In [1]:
import re
import csv
from typing import List, Dict, Tuple

In [2]:
def parse_format_string(format_str: str) -> List[Tuple[int, str]]:
    pattern = r'%-?(\d+)s'
    matches = re.findall(pattern, format_str)
    widths = [(int(width), 'left') for width in matches]
    return widths


def parse_header(header_line: str) -> List[str]:
    return [field.strip() for field in header_line.split(',')]


def parse_fixed_width_line(line: str, widths: List[Tuple[int, str]]) -> List[str]:
    fields = []
    position = 0

    for width, alignment in widths:
        if position >= len(line):
            fields.append('')
        else:
            field = line[position:position + width].strip()
            fields.append(field)
        position += width

    return fields


def parse_file(input_file: str, header: str, format_str: str, output_file: str | None):
    """
    Parse a fixed-width text file based on header and format specification.
    
    Args:
        input_file: Path to the input text file
        header: Comma-separated header string
        format_str: Format string (e.g., %-2s%-9s%-20s)
        output_file: Optional output CSV file path
    """
    # Parse header and format
    field_names = parse_header(header)
    widths = parse_format_string(format_str)

    # Validate that header and format match
    if len(field_names) != len(widths):
        raise ValueError(f"Header has {len(field_names)} fields but format has {len(widths)} fields")

    print(f"Processing file: {input_file}")
    print(f"Number of fields: {len(field_names)}")
    print(f"Field widths: {[w[0] for w in widths]}")
    print("-" * 80)

    # Parse the file
    parsed_data = []

    with open(input_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            # Skip empty lines
            if not line.strip():
                continue

            # Parse the line
            values = parse_fixed_width_line(line, widths)

            # Create a dictionary of field: value pairs
            record = dict(zip(field_names, values))
            parsed_data.append(record)

            # Print first few records as sample
            if line_num <= 3:
                print(f"Record {line_num}:")
                for field, value in record.items():
                    if value:  # Only show non-empty fields
                        print(f"  {field}: {value}")
                print()

    print(f"Total records parsed: {len(parsed_data)}")

    # Write to CSV if output file specified
    if output_file:
        with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=field_names)
            writer.writeheader()
            writer.writerows(parsed_data)
        print(f"Output written to: {output_file}")

    return parsed_data

In [4]:
# Your header and format specifications
header = """recordType,nricFin,foreignDocNumber,mtExemptionIndicator,foreignLang3rdLangIndicator,foreignLangLieuMtIndicator,subjPaperExemptionEntry,exceedMxSubjLimitIndicator,indexNumber,schoolCode,acadLevelCode,acadStreamCode,className,examLevelCode,numberOfAttempt,statutoryName,hanyuPinyinName,gender,raceCategory1,nationality,houseBlockNumber,storeyNumber,unitNumber,buildingName,streetName,postalCode,foreignAddress1,foreignAddress2,foreignAddress3,preferedAddress,validEntryIndicator,telephoneNo,schoolGrpCode,schoolTypeCode,cddtTypeCode,dateOfBirth,midYrMotherTongueSubj,totalNoOfSubjOffered,ageOfCddt,ageCode,PRStatus,schoolUEN,raceCategory2"""

format_str = """%-2s%-9s%-20s%-1s%-1s%-1s%-1s%-1s%-10s%-4s%-6s%-6s%-15s%-6s%-1s%-66s%-66s%-1s%-6s%-2s%-10s%-3s%-5s%-66s%-32s%-6s%-40s%-40s%-40s%-1s%-1s%-12s%-6s%-6s%-6s%-8s%-6s%-2s%-2s%-2s%-1s%-10s%-6s"""

# Example: Parse a file
# parsed_data = parse_file('input.txt', header, format_str, 'output.csv')

# For testing without a file:
print("Script ready to use!")
print("\nTo parse a file, use:")
print("parsed_data = parse_file('your_input_file.txt', header, format_str, 'output.csv')")

Script ready to use!

To parse a file, use:
parsed_data = parse_file('your_input_file.txt', header, format_str, 'output.csv')


In [6]:
data = parse_file('/Users/duclm/projects/logs/XE2_SEC_CDDTREG_K_20251219_160410.TXT', header, format_str, 'output.csv')

Processing file: /Users/duclm/projects/logs/XE2_SEC_CDDTREG_K_20251219_160410.TXT
Number of fields: 43
Field widths: [2, 9, 20, 1, 1, 1, 1, 1, 10, 4, 6, 6, 15, 6, 1, 66, 66, 1, 6, 2, 10, 3, 5, 66, 32, 6, 40, 40, 40, 1, 1, 12, 6, 6, 6, 8, 6, 2, 2, 2, 1, 10, 6]
--------------------------------------------------------------------------------
Record 1:
  recordType: 00
  nricFin: XE2
  foreignDocNumber: CANDIDATE
  mtExemptionIndicator: _
  foreignLang3rdLangIndicator: I
  foreignLangLieuMtIndicator: N
  subjPaperExemptionEntry: F
  exceedMxSubjLimitIndicator: O
  indexNumber: 2025
  schoolCode: 1208

Record 2:
  recordType: 01
  foreignDocNumber: T1900162D
  mtExemptionIndicator: N
  foreignLang3rdLangIndicator: N
  foreignLangLieuMtIndicator: N
  subjPaperExemptionEntry: N
  indexNumber: 28110023
  schoolCode: 3622
  acadLevelCode: SEC4
  acadStreamCode: N.A
  className: S4-E2
  examLevelCode: SEC
  numberOfAttempt: 2
  statutoryName: CDDT C1848143A
  hanyuPinyinName: Li Wei
  gender: F


In [16]:
m = {}

for datum in data:
    if datum['indexNumber'] not in m:
        m[datum['indexNumber']] = datum
        # print(datum['indexNumber'] + " not duplicate")
    else:
        print(datum['indexNumber'] + " duplicate")

In [9]:
len(m)

4308

In [10]:
header = """recordType,examYear,indexNumber,langMediumCode,subjectCode,subjectAlphaGrade,subjectNumericGrade,examSeriesCode,examAuthorityCode,oralGrade,higherAwardGrade,exitEligibility,subjectLevelcode,resitIndicator"""
format_str = """%-2s%-4s%-10s%-6s%-4s%-35s%-15s%-6s%-6s%-35s%-35s%-15s%-6s%-1s """
data2 = parse_file(
    '/Users/duclm/projects/logs/XE2_SEC_CDDT_SUBJ_RESULT_K_20251208_170433.TXT',
    header, format_str, 'output2.csv')

Processing file: /Users/duclm/projects/logs/XE2_SEC_CDDT_SUBJ_RESULT_K_20251208_170433.TXT
Number of fields: 14
Field widths: [2, 4, 10, 6, 4, 35, 15, 6, 6, 35, 35, 15, 6, 1]
--------------------------------------------------------------------------------
Record 1:
  recordType: 00
  examYear: XE2
  subjectCode: CAND
  subjectAlphaGrade: _SUBJ_RESULT    20251208

Record 2:
  recordType: 01
  examYear: 2025
  indexNumber: 20010001
  langMediumCode: ENGLSH
  subjectCode: K227
  subjectAlphaGrade: THREE
  examSeriesCode: YE
  examAuthorityCode: CAMB
  subjectLevelcode: G2
  resitIndicator: N

Record 3:
  recordType: 01
  examYear: 2025
  indexNumber: 20010001
  langMediumCode: ENGLSH
  subjectCode: K300
  subjectAlphaGrade: NINE
  examSeriesCode: YE
  examAuthorityCode: CAMB
  subjectLevelcode: G3
  resitIndicator: N

Total records parsed: 5238
Output written to: output2.csv


In [11]:
m2 = {}

for datum in data2:
    if datum['indexNumber'] not in m2:
        m2[datum['indexNumber']] = datum
        print(datum['indexNumber'] + " not duplicate")
    else:
        print(datum['indexNumber'] + " duplicate")

len(m2)

 not duplicate
20010001 not duplicate
20010001 duplicate
20010001 duplicate
20010001 duplicate
20010001 duplicate
20010001 duplicate
20010002 not duplicate
20010002 duplicate
20010002 duplicate
20010002 duplicate
20010002 duplicate
20010002 duplicate
20010003 not duplicate
20010003 duplicate
20010003 duplicate
20010003 duplicate
20010003 duplicate
20010003 duplicate
20010004 not duplicate
20010004 duplicate
20010004 duplicate
20010004 duplicate
20010004 duplicate
20010004 duplicate
20010005 not duplicate
20010005 duplicate
20010005 duplicate
20010005 duplicate
20010005 duplicate
20010005 duplicate
20010006 not duplicate
20010006 duplicate
20010006 duplicate
20010006 duplicate
20010006 duplicate
20010006 duplicate
20010007 not duplicate
20010007 duplicate
20010007 duplicate
20010007 duplicate
20010007 duplicate
20010007 duplicate
20010008 not duplicate
20010008 duplicate
20010008 duplicate
20010008 duplicate
20010008 duplicate
20010008 duplicate
20010009 not duplicate
20010009 duplicate

1697

In [15]:
for d in m2:
    if d in m:
        # print(f"{d}")
        pass
    else:
        print(f"{d} not found")


28410009 not found
28960006 not found
